# Sprint 3 — 12_experiment_tracking.ipynb
## Rol: Experiment Tracker

### Objetivo
Cerrar el Sprint 3 dejando el trabajo **reproducible, versionado y listo para Sprint 4**. Este notebook no compite con los notebooks de entrenamiento, evaluación o comparación; su función es **consolidar artefactos, validar que puedan recargarse, registrar experimentos y preparar la base reusable del sprint siguiente**.

### Qué sí hace este notebook
- Verifica que existan los outputs clave de PB-11, PB-12 y PB-13.
- Carga los modelos baseline persistidos.
- Valida que cada pipeline pueda recargarse y predecir.
- Construye / actualiza un registro de experimentos consolidado.
- Documenta qué artefactos deben quedar en Git y cuáles en DVC.
- Prepara la base para `src/models.py` como módulo reusable del proyecto.
- Genera un checklist final de cierre del Sprint 3.

### Qué NO hace este notebook
- No reentrena modelos.
- No recalcula métricas pesadas.
- No vuelve a correr tuning.
- No modifica los resultados finales ya obtenidos por PB-11, PB-12 y PB-13.
- No reemplaza el versionamiento con Git/DVC; lo documenta y valida.

### Entregables esperados del rol
- `src/models.py`
- `models/experiments_log.csv`
- validación de carga y predicción de modelos baseline
- checklist de artefactos del Sprint 3
- cierre reproducible para Sprint 4

### Pendientes arrastrados de etapas previas que este rol debe amarrar
- Los `.pkl` baseline deben seguir bajo **DVC**, no Git.
- Los outputs livianos (`.csv`, `.png`, notebooks) deben quedar en **Git**.
- No deben quedar archivos `*_partial.csv` como entregables finales.
- Debe quedar claro qué archivos son inputs obligatorios para Sprint 4.

## 1. Checklist antes de correr
Antes de ejecutar este notebook, valida:
1. Estar en la rama correcta del rol (`feat/PB-14` o equivalente).
2. Haber traído de `main` todo lo mergeado en PB-11, PB-12 y PB-13.
3. Haber hecho `dvc pull` para recuperar los `.pkl` baseline.
4. Tener disponible `src/preprocessing.py`.
5. Tener claro que este rol cierra el Sprint 3 y deja lista la base para Sprint 4.

In [6]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd

from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## 2. Configuración del proyecto
Ajusta solo si tu estructura cambió.

In [7]:
PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path(".").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures" / "sprint3"
SRC_DIR = PROJECT_ROOT / "src"

TARGET_COL = "IsBadBuy"

EXPERIMENT_LOG_PATH = MODELS_DIR / "experiments_log.csv"
SRC_MODELS_PATH = SRC_DIR / "models.py"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR exists:", SRC_DIR.exists())

PROJECT_ROOT: /Users/alexandralozano/dp261-g1
SRC_DIR exists: True


## 3. Inputs esperados de PB-11, PB-12 y PB-13
Este rol no debería avanzar si faltan outputs clave del sprint.

In [8]:
required_files = [
    DATA_DIR / "train_balanced.csv",
    MODELS_DIR / "baseline_manifest.csv",
    MODELS_DIR / "baseline_smoke_test.csv",
    MODELS_DIR / "baseline_lr.pkl",
    MODELS_DIR / "baseline_dt.pkl",
    MODELS_DIR / "baseline_rf.pkl",
    MODELS_DIR / "baseline_svm.pkl",
    MODELS_DIR / "baseline_knn.pkl",
    MODELS_DIR / "evaluation_cv_summary.csv",
    MODELS_DIR / "evaluation_cv_fold_results.csv",
    MODELS_DIR / "evaluation_oof_metrics.csv",
    MODELS_DIR / "evaluation_classification_reports.csv",
    MODELS_DIR / "model_comparison_summary.csv",
    MODELS_DIR / "model_selection_candidates.csv",
    MODELS_DIR / "model_discard_reasons.csv",
    REPORTS_DIR / "model_comparison_primary_metric.png",
    REPORTS_DIR / "model_comparison_tradeoffs.png",
    SRC_DIR / "preprocessing.py",
]

optional_files = [
    MODELS_DIR / "evaluation_train_test_gap_subset.csv",
    REPORTS_DIR / "roc_baselines.png",
    REPORTS_DIR / "pr_baselines.png",
    REPORTS_DIR / "confusion_matrices_baselines.png",
]

check_df = pd.DataFrame({
    "artifact_path": [str(p) for p in required_files + optional_files],
    "required": [True] * len(required_files) + [False] * len(optional_files),
    "exists": [p.exists() for p in required_files + optional_files],
})

display(check_df)

missing_required = check_df[(check_df["required"]) & (~check_df["exists"])]
if not missing_required.empty:
    raise FileNotFoundError(
        "Faltan artefactos obligatorios para cerrar Sprint 3: "
        + ", ".join(missing_required["artifact_path"].tolist())
    )

,artifact_path,required,exists
0,/Users/alexandralozano/dp261-g1/data/processed...,True,True
1,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
2,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
3,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
4,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
5,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
6,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
7,/Users/alexandralozano/dp261-g1/models/baselin...,True,True
8,/Users/alexandralozano/dp261-g1/models/evaluat...,True,True
9,/Users/alexandralozano/dp261-g1/models/evaluat...,True,True


## 4. Carga de insumos y revisión rápida

In [9]:
train_df = pd.read_csv(DATA_DIR / "train_balanced.csv")
baseline_manifest_df = pd.read_csv(MODELS_DIR / "baseline_manifest.csv")
baseline_smoke_df = pd.read_csv(MODELS_DIR / "baseline_smoke_test.csv")
eval_cv_summary_df = pd.read_csv(MODELS_DIR / "evaluation_cv_summary.csv")
eval_oof_df = pd.read_csv(MODELS_DIR / "evaluation_oof_metrics.csv")
comparison_df = pd.read_csv(MODELS_DIR / "model_comparison_summary.csv")
selected_df = pd.read_csv(MODELS_DIR / "model_selection_candidates.csv")

print("train_df:", train_df.shape)
print("baseline_manifest_df:", baseline_manifest_df.shape)
print("eval_cv_summary_df:", eval_cv_summary_df.shape)
print("comparison_df:", comparison_df.shape)
print("selected_df:", selected_df.shape)

display(baseline_manifest_df)
display(selected_df)

train_df: (102410, 60)
baseline_manifest_df: (5, 8)
eval_cv_summary_df: (5, 15)
comparison_df: (5, 42)
selected_df: (3, 42)


,model_key,estimator,artifact_path,target_col,train_rows,train_columns,status,notes
0,lr,LogisticRegression,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
1,dt,DecisionTreeClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
2,rf,RandomForestClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
3,svm,SVC,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters
4,knn,KNeighborsClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,Default baseline parameters


,rank,model_key,primary_metric,cv_folds,test_accuracy_mean,test_accuracy_std,test_precision_mean,test_precision_std,test_recall_mean,test_recall_std,test_f1_mean,test_f1_std,test_roc_auc_mean,test_roc_auc_std,fit_time_mean,score_time_mean,score_method,oof_accuracy,oof_precision,oof_recall,oof_f1,oof_roc_auc,oof_average_precision,train_accuracy_mean,accuracy_gap,train_precision_mean,precision_gap,train_recall_mean,recall_gap,train_f1_mean,f1_gap,train_roc_auc_mean,roc_auc_gap,f1_diagnosis,rank_primary_metric,interpretability,model_family,speed_bucket,precision_recall_gap,selected_for_sprint4,selection_reason,discard_reason
0,1,rf,f1,5,0.988653,0.000424,0.979625,0.000929,0.998067,0.000272,0.988759,0.000414,0.999437,0.000143,63.006968,1.162740,predict_proba,0.988653,0.979624,0.998067,0.988759,0.999437,0.999592,1.0,0.011347,1.0,0.020375,1.0,0.001933,1.0,0.011241,1.0,0.000563,sin brecha fuerte,1,media,ensamble de árboles,más lento,-0.018442,True,Top 1 por F1 (CV) | OOF F1=0.989 | AUC CV=0.99...,NaN
1,2,dt,f1,5,0.925134,0.001772,0.870759,0.002643,0.998477,0.000132,0.930252,0.001539,0.925134,0.001772,11.453154,0.365945,predict_proba,0.925134,0.870751,0.998477,0.930250,0.925134,0.870186,1.0,0.074866,1.0,0.129241,1.0,0.001523,1.0,0.069748,1.0,0.074866,brecha moderada,2,alta,árbol simple,más rápido,-0.127718,True,Top 2 por F1 (CV) | OOF F1=0.930 | AUC CV=0.92...,NaN
2,3,knn,f1,5,0.825144,0.001038,0.754863,0.001089,0.963031,0.002372,0.846332,0.000978,0.907716,0.000912,1.322752,234.480872,predict_proba,0.825144,0.754860,0.963031,0.846333,0.907709,0.855978,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,media-baja,instancias / distancia,más rápido,-0.208168,True,Top 3 por F1 (CV) | OOF F1=0.846 | AUC CV=0.90...,NaN


## 5. Validación de carga y predicción de pipelines baseline
Este paso es central para el rol: confirmar que los artefactos persistidos realmente sirven para reproducir resultados y para ser reutilizados en Sprint 4.

In [10]:
model_paths = {
    "lr": MODELS_DIR / "baseline_lr.pkl",
    "dt": MODELS_DIR / "baseline_dt.pkl",
    "rf": MODELS_DIR / "baseline_rf.pkl",
    "svm": MODELS_DIR / "baseline_svm.pkl",
    "knn": MODELS_DIR / "baseline_knn.pkl",
}

X_sample = train_df.drop(columns=[TARGET_COL]).head(5).copy()

validation_rows = []
loaded_models = {}

for model_key, path in model_paths.items():
    pipe = joblib.load(path)
    preds = pipe.predict(X_sample)

    loaded_models[model_key] = pipe
    validation_rows.append({
        "model_key": model_key,
        "artifact_path": str(path),
        "artifact_exists": path.exists(),
        "loaded_ok": True,
        "n_sample_predictions": len(preds),
        "sample_predictions": json.dumps(preds.tolist()),
    })

model_reload_validation_df = pd.DataFrame(validation_rows)
display(model_reload_validation_df)

,model_key,artifact_path,artifact_exists,loaded_ok,n_sample_predictions,sample_predictions
0,lr,/Users/alexandralozano/dp261-g1/models/baselin...,True,True,5,"[1, 1, 0, 1, 1]"
1,dt,/Users/alexandralozano/dp261-g1/models/baselin...,True,True,5,"[1, 1, 0, 0, 1]"
2,rf,/Users/alexandralozano/dp261-g1/models/baselin...,True,True,5,"[1, 1, 0, 0, 1]"
3,svm,/Users/alexandralozano/dp261-g1/models/baselin...,True,True,5,"[1, 1, 1, 1, 1]"
4,knn,/Users/alexandralozano/dp261-g1/models/baselin...,True,True,5,"[1, 1, 1, 0, 1]"


## 6. Consolidación del registro de experimentos
La idea es dejar un `experiments_log.csv` único con:
- metadata del modelo,
- métricas clave,
- si fue seleccionado o no,
- y notas útiles para Sprint 4.

In [11]:
comparison_cols = [
    "model_key",
    "rank",
    "selected_for_sprint4",
    "selection_reason",
    "discard_reason",
    "model_family",
    "interpretability",
    "speed_bucket",
]
comparison_cols_existing = [c for c in comparison_cols if c in comparison_df.columns]

experiments_df = baseline_manifest_df.merge(
    eval_cv_summary_df,
    on="model_key",
    how="left"
).merge(
    eval_oof_df,
    on="model_key",
    how="left"
).merge(
    comparison_df[comparison_cols_existing],
    on="model_key",
    how="left"
)

experiments_df["sprint"] = 3
experiments_df["stage"] = "modeling_baseline"
experiments_df["log_timestamp"] = datetime.now().isoformat(timespec="seconds")

if "status" not in experiments_df.columns:
    experiments_df["status"] = "tracked"

preferred_cols = [
    "log_timestamp",
    "sprint",
    "stage",
    "model_key",
    "estimator",
    "artifact_path",
    "target_col",
    "train_rows",
    "train_columns",
    "status",
    "rank",
    "selected_for_sprint4",
    "selection_reason",
    "discard_reason",
    "primary_metric",
    "test_accuracy_mean",
    "test_precision_mean",
    "test_recall_mean",
    "test_f1_mean",
    "test_roc_auc_mean",
    "oof_accuracy",
    "oof_precision",
    "oof_recall",
    "oof_f1",
    "oof_roc_auc",
    "oof_average_precision",
    "fit_time_mean",
    "score_time_mean",
    "model_family",
    "interpretability",
    "speed_bucket",
    "notes",
]
experiments_cols_existing = [c for c in preferred_cols if c in experiments_df.columns]
experiments_df = experiments_df[experiments_cols_existing].copy()

display(experiments_df)

,log_timestamp,sprint,stage,model_key,estimator,artifact_path,target_col,train_rows,train_columns,status,rank,selected_for_sprint4,selection_reason,discard_reason,primary_metric,test_accuracy_mean,test_precision_mean,test_recall_mean,test_f1_mean,test_roc_auc_mean,oof_accuracy,oof_precision,oof_recall,oof_f1,oof_roc_auc,oof_average_precision,fit_time_mean,score_time_mean,model_family,interpretability,speed_bucket,notes
0,2026-04-23T17:59:25,3,modeling_baseline,lr,LogisticRegression,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,5,False,NaN,No quedó en top 3 por F1 (CV),f1,0.663441,0.665506,0.657319,0.661351,0.724553,0.663441,0.665467,0.657319,0.661368,0.724474,0.708676,10.076349,0.384313,lineal,alta,más rápido,Default baseline parameters
1,2026-04-23T17:59:25,3,modeling_baseline,dt,DecisionTreeClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,2,True,Top 2 por F1 (CV) | OOF F1=0.930 | AUC CV=0.92...,NaN,f1,0.925134,0.870759,0.998477,0.930252,0.925134,0.925134,0.870751,0.998477,0.930250,0.925134,0.870186,11.453154,0.365945,árbol simple,alta,más rápido,Default baseline parameters
2,2026-04-23T17:59:25,3,modeling_baseline,rf,RandomForestClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,1,True,Top 1 por F1 (CV) | OOF F1=0.989 | AUC CV=0.99...,NaN,f1,0.988653,0.979625,0.998067,0.988759,0.999437,0.988653,0.979624,0.998067,0.988759,0.999437,0.999592,63.006968,1.162740,ensamble de árboles,media,más lento,Default baseline parameters
3,2026-04-23T17:59:25,3,modeling_baseline,svm,SVC,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,4,False,NaN,No quedó en top 3 por F1 (CV) | mayor costo co...,f1,0.499502,0.499749,0.994122,0.665131,0.560083,0.499502,0.499750,0.994122,0.665134,0.511980,0.520305,18.243641,7.023308,margen máximo,baja,más lento,Default baseline parameters
4,2026-04-23T17:59:25,3,modeling_baseline,knn,KNeighborsClassifier,/Users/alexandralozano/dp261-g1/models/baselin...,IsBadBuy,102410,59,trained,3,True,Top 3 por F1 (CV) | OOF F1=0.846 | AUC CV=0.90...,NaN,f1,0.825144,0.754863,0.963031,0.846332,0.907716,0.825144,0.754860,0.963031,0.846333,0.907709,0.855978,1.322752,234.480872,instancias / distancia,media-baja,más rápido,Default baseline parameters


## 7. Persistencia del `experiments_log.csv`
Si el archivo ya existe, aquí puedes decidir si:
- sobrescribirlo con la versión consolidada del Sprint 3, o
- concatenar con logs futuros.

Para cerrar Sprint 3, lo más claro es sobrescribir con la versión consolidada actual.

In [12]:
experiments_df.to_csv(EXPERIMENT_LOG_PATH, index=False)
print("Saved:", EXPERIMENT_LOG_PATH)

Saved: /Users/alexandralozano/dp261-g1/models/experiments_log.csv


## 8. Plantilla lógica para `src/models.py`
Este notebook no tiene por qué escribir código complejo automáticamente, pero sí debe dejar claro qué funciones mínimas necesita el módulo reusable para Sprint 4.

In [13]:
src_models_blueprint = {
    "module_path": str(SRC_MODELS_PATH),
    "expected_functions": [
        "load_model_artifact",
        "save_model_artifact",
        "predict_with_model",
        "load_experiment_log",
        "append_experiment_log",
        "validate_model_artifact",
    ],
    "purpose": "Consolidar lógica reusable de persistencia, validación y logging para Sprint 4"
}

display(pd.DataFrame({
    "key": src_models_blueprint.keys(),
    "value": [str(v) for v in src_models_blueprint.values()]
}))

,key,value
0,module_path,/Users/alexandralozano/dp261-g1/src/models.py
1,expected_functions,"['load_model_artifact', 'save_model_artifact',..."
2,purpose,"Consolidar lógica reusable de persistencia, va..."


## 9. Checklist final de cierre del Sprint 3
Este bloque resume si el sprint ya quedó realmente cerrrado desde la perspectiva de artefactos y reproducibilidad.

In [14]:
sprint3_closure_checks = [
    ("09_baseline_models.ipynb disponible", (PROJECT_ROOT / "notebooks" / "09_baseline_models.ipynb").exists()),
    ("10_evaluation.ipynb disponible", (PROJECT_ROOT / "notebooks" / "10_evaluation.ipynb").exists()),
    ("11_model_comparison.ipynb disponible", (PROJECT_ROOT / "notebooks" / "11_model_comparison.ipynb").exists()),
    ("Modelos baseline disponibles", all(path.exists() for path in model_paths.values())),
    ("evaluation_cv_summary.csv disponible", (MODELS_DIR / "evaluation_cv_summary.csv").exists()),
    ("model_comparison_summary.csv disponible", (MODELS_DIR / "model_comparison_summary.csv").exists()),
    ("experiments_log.csv generado", EXPERIMENT_LOG_PATH.exists()),
    ("Validación de recarga de modelos hecha", not model_reload_validation_df.empty),
    ("Hay candidatos seleccionados para Sprint 4", "selected_for_sprint4" in comparison_df.columns and comparison_df["selected_for_sprint4"].fillna(False).any()),
]

sprint3_closure_df = pd.DataFrame(sprint3_closure_checks, columns=["check", "passed"])
display(sprint3_closure_df)

,check,passed
0,09_baseline_models.ipynb disponible,True
1,10_evaluation.ipynb disponible,True
2,11_model_comparison.ipynb disponible,True
3,Modelos baseline disponibles,True
4,evaluation_cv_summary.csv disponible,True
5,model_comparison_summary.csv disponible,True
6,experiments_log.csv generado,True
7,Validación de recarga de modelos hecha,True
8,Hay candidatos seleccionados para Sprint 4,True


## 10. Versionamiento: Git vs DVC
Este rol debe dejar explícito qué va en Git y qué sigue en DVC.

In [15]:
versioning_df = pd.DataFrame([
    {"artifact_type": "notebooks (.ipynb)", "recommended_tracking": "Git", "notes": "Outputs livianos y revisables"},
    {"artifact_type": "csv de evaluación/comparación/log", "recommended_tracking": "Git", "notes": "Outputs livianos; no usar *_partial.csv"},
    {"artifact_type": "figuras (.png)", "recommended_tracking": "Git", "notes": "Salvo que crezcan demasiado"},
    {"artifact_type": "modelos baseline (.pkl)", "recommended_tracking": "DVC", "notes": "Pesados; ya versionados desde PB-11"},
    {"artifact_type": "archivos .dvc", "recommended_tracking": "Git", "notes": "Punteros del artefacto DVC"},
    {"artifact_type": "outputs temporales / partial", "recommended_tracking": "No incluir", "notes": "Eliminar o dejar fuera del PR final"},
])

display(versioning_df)

,artifact_type,recommended_tracking,notes
0,notebooks (.ipynb),Git,Outputs livianos y revisables
1,csv de evaluación/comparación/log,Git,Outputs livianos; no usar *_partial.csv
2,figuras (.png),Git,Salvo que crezcan demasiado
3,modelos baseline (.pkl),DVC,Pesados; ya versionados desde PB-11
4,archivos .dvc,Git,Punteros del artefacto DVC
5,outputs temporales / partial,No incluir,Eliminar o dejar fuera del PR final


## 11. Revisión de artefactos generados por este rol

In [16]:
generated_paths = [
    EXPERIMENT_LOG_PATH,
]

generated_df = pd.DataFrame({
    "artifact_path": [str(p) for p in generated_paths],
    "exists": [p.exists() for p in generated_paths],
    "size_kb": [round(p.stat().st_size / 1024, 2) if p.exists() else None for p in generated_paths],
})

display(generated_df)

,artifact_path,exists,size_kb
0,/Users/alexandralozano/dp261-g1/models/experim...,True,3.08


## 12. Handoff y cierre del sprint
### Qué deja listo este rol
- `experiments_log.csv` consolidado
- validación de recarga de pipelines baseline
- criterios claros de versionamiento Git vs DVC
- checklist de cierre del Sprint 3
- base documental para `src/models.py` y Sprint 4

### Qué debería existir al terminar el Sprint 3
- notebooks 09, 10 y 11 funcionales
- modelos baseline persistidos y bajo DVC
- outputs de evaluación y comparación en Git
- candidatos seleccionados para Sprint 4
- módulo reusable `src/models.py`
- `models/experiments_log.csv`

### Qué sigue en Sprint 4
- tuning de hiperparámetros sobre los candidatos seleccionados
- modelado avanzado
- comparación contra estos baselines ya priorizados